# Fine-tune Domain-Adapted IndoBERT on Source Dataset

This notebook fine-tunes the domain-adapted IndoBERT (from MLM training) for sentiment classification using source_balanced.csv

## Import Libraries

In [ ]:
import pandas as pd
import torch
import numpy as np
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Load Source Dataset

In [ ]:
# Load source_balanced.csv
df = pd.read_csv('source_balanced.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nLabel distribution:")
print(df['label'].value_counts())
print(f"\nFirst few rows:")
df.head()

In [ ]:
# Create label mapping
label2id = {'negative': 0, 'positive': 1}
id2label = {0: 'negative', 1: 'positive'}

# Convert labels to numeric
df['label_id'] = df['label'].map(label2id)

print("Label mapping:")
print(f"  negative → 0")
print(f"  positive → 1")
print(f"\nLabel distribution (numeric):")
print(df['label_id'].value_counts().sort_index())

## Load Domain-Adapted Model

Load the IndoBERT model that was adapted to the target domain via MLM training

In [ ]:
# Load the domain-adapted model
model_path = "./indobert_mlm_target_final"

print(f"Loading domain-adapted model from: {model_path}")

# Load tokenizer (same as before)
tokenizer = AutoTokenizer.from_pretrained(model_path)

# Load model for sequence classification (not MLM)
# This will add a classification head on top of the MLM-adapted base model
model = AutoModelForSequenceClassification.from_pretrained(
    model_path,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True  # Ignore the MLM head, use classification head instead
)

print(f"Model loaded successfully!")
print(f"Model type: {model.__class__.__name__}")
print(f"Number of labels: {model.config.num_labels}")

## Prepare Data for Training

In [ ]:
# Create dataset from DataFrame
dataset_dict = {
    "text": df['content'].tolist(),
    "label": df['label_id'].tolist()
}

dataset = Dataset.from_dict(dataset_dict)

print(f"Dataset size: {len(dataset)}")
print(f"Dataset features: {dataset.features}")
print(f"\nFirst example:")
print(dataset[0])

In [ ]:
# Tokenize the dataset
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        padding=False  # Dynamic padding will be handled by data collator
    )

print("Tokenizing dataset...")
tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
    desc="Tokenizing"
)

print(f"Tokenization complete!")
print(f"Tokenized dataset features: {tokenized_dataset.features}")

In [ ]:
# Split dataset into train and test (80/20 split)
train_test_split = tokenized_dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = train_test_split['train']
test_dataset = train_test_split['test']

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

In [ ]:
# Create data collator for dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("Data collator created for dynamic padding")

## Define Evaluation Metrics

In [ ]:
# Define compute_metrics function for evaluation
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    
    # Calculate metrics
    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='weighted'
    )
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

print("Evaluation metrics function defined")

## Configure Training Parameters

In [ ]:
# Define training arguments
training_args = TrainingArguments(
    output_dir="./indobert_source_finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=500,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    push_to_hub=False,
    report_to="none"
)

print("Training configuration:")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  FP16: {training_args.fp16}")

In [ ]:
# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=tokenizer
)

print("Trainer initialized successfully!")

## Train the Model

In [ ]:
# Start training
print("Starting fine-tuning on source domain data...")
print("=" * 60)

train_result = trainer.train()

print("\n" + "=" * 60)
print("Training completed!")
print(f"Training loss: {train_result.training_loss:.4f}")
print(f"Training time: {train_result.metrics['train_runtime']:.2f} seconds")

## Evaluate the Model

In [ ]:
# Evaluate on test set
print("Evaluating model on test set...")
eval_results = trainer.evaluate()

print("\nTest Set Results:")
print(f"  Accuracy:  {eval_results['eval_accuracy']:.4f}")
print(f"  Precision: {eval_results['eval_precision']:.4f}")
print(f"  Recall:    {eval_results['eval_recall']:.4f}")
print(f"  F1 Score:  {eval_results['eval_f1']:.4f}")
print(f"  Loss:      {eval_results['eval_loss']:.4f}")

In [ ]:
# Get detailed predictions for classification report
predictions = trainer.predict(test_dataset)
pred_labels = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids

# Print detailed classification report
print("\nDetailed Classification Report:")
print("="*60)
print(classification_report(
    true_labels, 
    pred_labels, 
    target_names=['negative', 'positive'],
    digits=4
))

## Save the Fine-tuned Model

In [ ]:
# Save the fine-tuned model
output_dir = "./indobert_source_final"

print(f"Saving model to {output_dir}...")
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

# Verify saved files
import os
if os.path.exists(output_dir):
    files = os.listdir(output_dir)
    print(f"\nSaved files in {output_dir}:")
    for file in files:
        file_path = os.path.join(output_dir, file)
        if os.path.isfile(file_path):
            size_mb = os.path.getsize(file_path) / (1024 * 1024)
            print(f"  - {file:30s} ({size_mb:.2f} MB)")

print(f"\n✓ Model and tokenizer saved successfully to: {output_dir}")
print(f"\nTo load the model later, use:")
print(f"  from transformers import AutoModelForSequenceClassification, AutoTokenizer")
print(f"  model = AutoModelForSequenceClassification.from_pretrained('{output_dir}')")
print(f"  tokenizer = AutoTokenizer.from_pretrained('{output_dir}')")

## Test Predictions (Optional)

Test the model with sample reviews

In [ ]:
# Test with sample reviews
from transformers import pipeline

# Create sentiment analysis pipeline
sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model=output_dir,
    tokenizer=output_dir
)

# Test samples
test_samples = [
    "Aplikasinya sangat membantu dan mudah digunakan!",
    "Pengiriman cepat, barang sesuai deskripsi. Sangat puas!",
    "Sangat kecewa dengan pelayanan. Tidak profesional!",
    "Aplikasi sering error dan lambat. Tolong diperbaiki.",
]

print("Test Predictions:")
print("="*60)
for i, text in enumerate(test_samples, 1):
    result = sentiment_analyzer(text)[0]
    label = result['label'].upper()
    score = result['score']
    
    print(f"\n{i}. Text: {text}")
    print(f"   Prediction: {label} (confidence: {score:.4f})")